# 01.2 Tensor Shape Ops

The core task of this notebook is to build genuine shape thinking. Many PyTorch bugs are not mathematical errors; they happen because a tensor has the wrong shape, the wrong axis order, or an unexpected singleton dimension.

You will practice reshaping, flattening, adding and removing dimensions, transposing axes, permuting axes, and using broadcasting. The goal is to know what each operation does to the meaning of each dimension, not just to memorize function names.

## Learning Goals

After this notebook, you should be able to:

1. Use `reshape`, `view`, and `flatten` comfortably.
2. Understand `unsqueeze` and `squeeze`.
3. Distinguish `transpose` from `permute`.
4. Predict output shapes after common operations.
5. Explain broadcasting using shapes.
6. Debug shape mismatch errors faster.

In [ ]:
import torch

## `reshape`, `view`, and `flatten`

These operations change how tensor dimensions are presented. `reshape` is the most convenient general tool when the total number of elements stays the same. `view` is similar, but it requires a compatible memory layout and can fail after operations such as transpose. `flatten` collapses a range of dimensions into one dimension.

The key rule is that shape changes do not create or destroy values. A tensor with 24 elements can become `(6, 4)` or `(2, 12)`, but it cannot become `(5, 5)` without changing the number of elements.

In [ ]:
x = torch.arange(24)
a = x.reshape(2, 3, 4)
b = a.flatten()
c = a.flatten(start_dim=1)

print("x.shape =", x.shape)
print("a.shape =", a.shape)
print("b.shape =", b.shape)
print("c.shape =", c.shape)

`flatten(start_dim=1)` is very common in neural networks.

For example:

- input is `(batch, channels, height, width)`
- after flattening it becomes `(batch, features)`

This usually happens before feeding convolution outputs into linear layers.


In [ ]:
# Exercise 1
#
# Start with x.shape == (2, 3, 4).
#
# Fill in:
# - x1: reshape x to shape (6, 4).
# - x2: flatten x so that the first dimension stays as batch size 2 and the
#   remaining dimensions become one feature dimension, giving shape (2, 12).

x = torch.arange(24).reshape(2, 3, 4)

# x1 =
# x2 =
# print(x1.shape)
# print(x2.shape)

In [ ]:
# Exercise 1 Reference Solution

x = torch.arange(24).reshape(2, 3, 4)
x1 = x.reshape(6, 4)
x2 = x.flatten(start_dim=1)
print(x1.shape)
print(x2.shape)

## `unsqueeze` and `squeeze`

`unsqueeze` inserts a dimension of size 1. This is useful when you need an explicit batch dimension or when you want broadcasting to happen along a specific axis. `squeeze` removes dimensions of size 1. It is useful after an operation temporarily adds an extra singleton dimension.

Be careful with `squeeze()` without specifying a dimension, because it removes all size-1 dimensions. In model code, specifying the dimension is often safer.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
x_row = x.unsqueeze(0)
x_col = x.unsqueeze(1)
x_back = x_col.squeeze(1)

print("x.shape =", x.shape)
print("x_row.shape =", x_row.shape)
print("x_col.shape =", x_col.shape)
print("x_back.shape =", x_back.shape)

These operations commonly appear when you need to manually create a batch dimension, control the direction of broadcasting, or match the input interface expected by a layer. In each case, a size-1 dimension is not meaningless; it tells PyTorch how to align the tensor with another tensor or layer.

In [ ]:
# Exercise 2
#
# Start with x.shape == (4,).
#
# Fill in:
# - a: add a dimension at the front so the shape becomes (1, 4).
# - b: add a dimension at the end so the shape becomes (4, 1).
# - c: remove the second dimension from b so the shape becomes (4,) again.

x = torch.arange(4)

# a =
# b =
# c =
# print(a.shape, b.shape, c.shape)

In [ ]:
# Exercise 2 Reference Solution

x = torch.arange(4)
a = x.unsqueeze(0)
b = x.unsqueeze(1)
c = b.squeeze(1)
print(a.shape, b.shape, c.shape)

## `transpose` and `permute`

Both operations reorder dimensions. `transpose(dim0, dim1)` swaps two dimensions. `permute(...)` gives a complete new ordering for all dimensions. Use `transpose` for a simple swap and `permute` when you need to move several axes at once.

This comes up constantly with images because PyTorch commonly uses channel-first image tensors, while plotting libraries often expect channel-last images.

In [ ]:
x = torch.arange(24).reshape(2, 3, 4)
xt = x.transpose(1, 2)
xp = x.permute(2, 0, 1)

print("x.shape =", x.shape)
print("xt.shape =", xt.shape)
print("xp.shape =", xp.shape)

Common image shape pattern:

- `PyTorch` commonly use `(batch, channels, height, width)`
- some external libraries often use `(height, width, channels)`

This is one reason why `permute` appears so often.


In [ ]:
# Exercise 3
#
# image.shape == (3, 32, 32), where the dimensions mean
# (channels, height, width).
#
# Convert it to image_hwc with shape (32, 32, 3), where the dimensions mean
# (height, width, channels). Use permute because all three axes need to be
# reordered.

image = torch.randn(3, 32, 32)

# image_hwc =
# print(image_hwc.shape)

In [ ]:
# Exercise 3 Reference Solution

image = torch.randn(3, 32, 32)
image_hwc = image.permute(1, 2, 0)
print(image_hwc.shape)

## Broadcasting

Broadcasting is dimension alignment, not random automatic expansion. PyTorch compares shapes from the last dimension backward. Each pair of dimensions must match, or one of them must be `1`. If the rule holds, PyTorch behaves as if the smaller tensor were repeated along the size-1 dimension.

This is why keeping dimensions with `keepdim=True` is often helpful: it preserves a size-1 axis that can broadcast cleanly against the original tensor.

In [ ]:
x = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
offset = torch.tensor([10.0, 100.0])
row_scale = torch.tensor([[1.0], [10.0], [100.0]])

print("x + offset =\n", x + offset)
print()
print("x * row_scale =\n", x * row_scale)

How to read it:

- `x.shape == (3, 2)`
- broadcasts row-wise

In [ ]:
# Exercise 4
#
# Implement center_by_column(x).
#
# Input:
# - x is a 2D tensor with shape (num_rows, num_columns).
#
# What to do:
# - Compute each column's mean.
# - Subtract the matching column mean from each value.
# - Return the centered tensor.
#
# Shape hint:
# - Use dim=0 to compute column means.
# - Use keepdim=True so the mean has shape (1, num_columns) and broadcasts
#   against x.

def center_by_column(x):
    # TODO
    pass


# sample = torch.tensor([[1.0, 10.0], [2.0, 20.0], [3.0, 30.0]])
# print(center_by_column(sample))

In [ ]:
# Exercise 4 Reference Solution

def center_by_column_solution(x):
    col_mean = x.mean(dim=0, keepdim=True)
    return x - col_mean


sample = torch.tensor([[1.0, 10.0], [2.0, 20.0], [3.0, 30.0]])
print(center_by_column_solution(sample))

## Non-Contiguous Tensors

This is a practical but often overlooked topic.

After operations like `transpose` or `permute`, a tensor may become non-contiguous.

In such cases:

- `reshape` is usually safer
- `view` may fail

In [ ]:
x = torch.arange(24).reshape(2, 3, 4)
xt = x.transpose(1, 2)

print("xt.is_contiguous() =", xt.is_contiguous())

try:
    bad = xt.view(2, 12)
    print("view result shape =", bad.shape)
except RuntimeError as e:
    print("view failed / view failed:", e)

good = xt.reshape(2, 12)
print("reshape result shape =", good.shape)

In [ ]:
# Exercise 5
#
# Start with x.shape == (2, 3, 4).
#
# Step 1:
# - Transpose dimensions 1 and 2 so y.shape becomes (2, 4, 3).
#
# Step 2:
# - Reshape y so z.shape becomes (2, 12).
#
# This exercise is meant to make you notice that a transpose changes axis order
# before the reshape collapses the last two dimensions.

x = torch.arange(24).reshape(2, 3, 4)

# y =
# z =
# print(y.shape)
# print(z.shape)

In [ ]:
# Exercise 5 Reference Solution

x = torch.arange(24).reshape(2, 3, 4)
y = x.transpose(1, 2)
z = y.reshape(2, 12)
print(y.shape)
print(z.shape)

## Summary

You should now start building a fixed habit:

1. first write down the input shape
2. then predict the output shape
3. finally run the code to verify

You should now be able to answer:

1. What problems do `reshape`, `flatten`, and `unsqueeze` solve?
2. What is the difference between `transpose` and `permute`?
3. Why does `view` sometimes fail while `reshape` works?
4. What is the minimal rule for broadcasting?

Suggested next step:

- Move to `01_03_autograd.ipynb` to understand how gradients are tracked and propagated.